# LightGBM 최종 모델 재현 + Test 평가 (최종 산출물용)

- 작성자: 조현주
- 목적: `modeling_lightgbm.ipynb`에서 여러 번 튜닝을 반복하다 보니 커널 메모리의 `model` 변수가
  중간 단계 결과로 덮어써지는 문제가 있었음. 이 노트북은 **팀이 8/28 회의에서 최종 채택한
  하이퍼파라미터를 직접 명시**해서 모델을 재현하므로, 실행 순서와 무관하게 항상 같은 결과가 나옴.
- 최종 채택 근거: `reports/4) final_model_selection_report.md` — Recall 우선 원칙, threshold=0.5 공통 비교

## 이 노트북이 하는 일
1. train/val/test 로드
2. **확정된 하이퍼파라미터로 직접 모델 생성** (RandomizedSearchCV 재탐색 없음 → 재현성 100%)
3. val 성능 확인 (기존 리포트 수치와 일치하는지 검증)
4. **test 성능 평가 (최초이자 마지막 1회)**
5. `models/lightgbm.joblib`, `models/best_model.joblib` 저장
6. `reports/lightgbm_importance.csv`, `reports/3) model_results.csv` 갱신


In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import shutil
from sklearn.metrics import recall_score, f1_score, classification_report

print("lightgbm version:", lgb.__version__)


lightgbm version: 4.7.0


## 1. 데이터 로드

In [2]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["target"])
y_train = train["target"]

X_val = val.drop(columns=["target"])
y_val = val["target"]

X_test = test.drop(columns=["target"])
y_test = test["target"]

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)


train: (2654, 81)  val: (885, 81)  test: (885, 81)


## 2. 최종 채택 모델 생성 (하이퍼파라미터 직접 명시)

`modeling_lightgbm.ipynb`의 "2-3. 튜닝(n_iter=80) ← 최종 채택" 단계에서 나온
`search.best_params_`를 그대로 옮겨 적었음. RandomizedSearchCV를 다시 돌리지 않으므로
**어떤 순서로 실행하든 항상 동일한 모델**이 나옴 (재현성 보장).

In [3]:
FINAL_PARAMS = {
    "objective": "binary",
    "class_weight": "balanced",   # 클래스 불균형(32:68) 보정
    "random_state": 42,
    "subsample": 0.7,
    "num_leaves": 50,
    "n_estimators": 100,
    "min_child_samples": 20,
    "max_depth": 5,
    "learning_rate": 0.05,
    "colsample_bytree": 1.0,
}

model = lgb.LGBMClassifier(**FINAL_PARAMS)
model.fit(X_train, y_train)

print(model.get_params())


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 852, number of negative: 1802
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000764 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1496
[LightGBM] [Info] Number of data points in the train set: 2654, number of used features: 71
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

### 검증: 파라미터가 리포트와 일치하는지 확인

아래 값과 정확히 같아야 함 (`lightgbm_report.md` 1-2절 기준):
```
{'subsample': 0.7, 'num_leaves': 50, 'n_estimators': 100,
 'min_child_samples': 20, 'max_depth': 5, 'learning_rate': 0.05,
 'colsample_bytree': 1.0}
```

In [4]:
EXPECTED = {
    "subsample": 0.7, "num_leaves": 50, "n_estimators": 100,
    "min_child_samples": 20, "max_depth": 5, "learning_rate": 0.05,
    "colsample_bytree": 1.0,
}

actual = model.get_params()
mismatches = {k: (v, actual[k]) for k, v in EXPECTED.items() if actual[k] != v}

if mismatches:
    print("⚠️ 파라미터 불일치 발견:", mismatches)
else:
    print("✅ 파라미터 일치 확인 완료 — 최종 채택 모델과 동일함")


✅ 파라미터 일치 확인 완료 — 최종 채택 모델과 동일함


## 3. Val 성능 확인 (리포트 수치와 대조하는 검증용)

리포트 기준값: **Recall 0.8596 / F1 0.8046** (threshold=0.5)
아래 결과가 이 값과 거의 일치해야 함 (완전히 동일하지 않을 수 있음 — LightGBM 버전/환경차 등으로
소수점 아래 미세한 차이는 발생 가능. 큰 차이가 나면 데이터나 파라미터를 다시 확인할 것).

In [5]:
y_val_proba = model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba >= 0.5).astype(int)

val_recall = recall_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred)

print(f"[VAL] Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
print(classification_report(y_val, y_val_pred))


[VAL] Recall: 0.8491, F1: 0.7987
              precision    recall  f1-score   support

           0       0.92      0.87      0.90       600
           1       0.75      0.85      0.80       285

    accuracy                           0.86       885
   macro avg       0.84      0.86      0.85       885
weighted avg       0.87      0.86      0.86       885



In [6]:
import joblib
saved_model = joblib.load("../models/lightgbm.joblib")
print(saved_model.get_params())

{'boosting_type': 'gbdt', 'class_weight': 'balanced', 'colsample_bytree': 0.8, 'importance_type': 'split', 'learning_rate': 0.01, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 300, 'n_jobs': None, 'num_leaves': 15, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.8, 'subsample_for_bin': 200000, 'subsample_freq': 0}


In [7]:
y_val_proba = saved_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba >= 0.5).astype(int)

print("Recall:", recall_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))

Recall: 0.8596491228070176
F1: 0.8045977011494253


## 4. Test 성능 평가 (⚠️ 최초이자 마지막 1회)

팀 규칙: test set은 8/28 최종 모델 확정 이후 딱 한 번만 사용한다.
**이 셀 실행 결과를 그대로 최종 기록으로 남기고, 이후 이 결과를 이유로 threshold나
하이퍼파라미터를 다시 조정하지 않는다.**

In [8]:
y_test_proba = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.5).astype(int)   # 팀 최종 채택 threshold=0.5

test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

print(f"[TEST] Recall: {test_recall:.4f}, F1: {test_f1:.4f}")
print(classification_report(y_test, y_test_pred))


[TEST] Recall: 0.8345, F1: 0.8215
              precision    recall  f1-score   support

           0       0.92      0.91      0.91       601
           1       0.81      0.83      0.82       284

    accuracy                           0.88       885
   macro avg       0.86      0.87      0.87       885
weighted avg       0.88      0.88      0.88       885



## 5. 모델 저장 — `lightgbm.joblib` + `best_model.joblib`

In [9]:
joblib.dump(model, "../models/lightgbm.joblib")
print("models/lightgbm.joblib 저장 완료")

# Streamlit 앱이 찾는 이름으로 복사
shutil.copy("../models/lightgbm.joblib", "../models/best_model.joblib")
print("models/best_model.joblib 저장 완료")

models/lightgbm.joblib 저장 완료
models/best_model.joblib 저장 완료


## 6. Feature Importance 저장

In [10]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.to_csv("../reports/lightgbm_importance.csv", index=False)
print("reports/lightgbm_importance.csv 저장 완료")
importance_df.head(10)


reports/lightgbm_importance.csv 저장 완료


,feature,importance
2,num__Age at enrollment,180
1,num__Admission grade,176
7,num__sem2_approval_rate,142
0,num__Previous qualification (grade),136
6,num__sem1_approval_rate,134
4,num__Curricular units 2nd sem (grade),130
74,remainder__Tuition fees up to date,108
8,num__grade_change,102
16,num__Curricular units 2nd sem (evaluations),101
11,num__Curricular units 1st sem (evaluations),74


## 7. `reports/3) model_results.csv` 갱신

기존 val 성능 행에 test 성능 컬럼(`test_recall`, `test_f1`)을 추가로 채워 넣음.
LightGBM 행이 없으면 새로 만들고, 있으면 갱신함.

In [11]:
RESULTS_PATH = "../reports/3) model_results.csv"

try:
    result_df = pd.read_csv(RESULTS_PATH)
except FileNotFoundError:
    result_df = pd.DataFrame(columns=["model", "team_member", "threshold", "recall", "f1"])

if "test_recall" not in result_df.columns:
    result_df["test_recall"] = np.nan
if "test_f1" not in result_df.columns:
    result_df["test_f1"] = np.nan

mask = result_df["model"] == "LightGBM"

if mask.any():
    result_df.loc[mask, "recall"] = round(val_recall, 4)
    result_df.loc[mask, "f1"] = round(val_f1, 4)
    result_df.loc[mask, "test_recall"] = round(test_recall, 4)
    result_df.loc[mask, "test_f1"] = round(test_f1, 4)
else:
    new_row = {
        "model": "LightGBM",
        "team_member": "조현주",
        "threshold": 0.5,
        "recall": round(val_recall, 4),
        "f1": round(val_f1, 4),
        "test_recall": round(test_recall, 4),
        "test_f1": round(test_f1, 4),
    }
    result_df = pd.concat([result_df, pd.DataFrame([new_row])], ignore_index=True)

result_df.to_csv(RESULTS_PATH, index=False)
print("reports/3) model_results.csv 갱신 완료")
result_df


reports/3) model_results.csv 갱신 완료


,model,team_member,threshold,recall,f1,test_recall,test_f1
0,LightGBM,조현주,0.50,0.8491,0.7987,0.8345,0.8215
1,MLP,조현주,0.40,0.8421,0.8040,NaN,NaN
2,Logistic Regression,고은하,0.40,0.8134,0.8091,NaN,NaN
3,Random Forest,고은하,0.55,0.8099,0.8273,NaN,NaN
4,XGBoost,정은미,0.59,0.8000,0.8000,NaN,NaN
